# Nested Hierarchical Structure with MQNLI

In this notebook, we use `pyvene` to analyze language models on a complex heirarchical task: multiply-quantified natural language inference (MQNLI). 
We begin by describing the causal structure that generates our data through semantic composition. Then, we analyze whether models that learn this task employ the same compositional structure to solve it.

In [10]:
__author__ = "Amir Zur"

## Set-Up

In [ ]:
# Colab/local setup for this MQNLI notebook.
# Run this cell once before the imports below, then restart the kernel/runtime.
import os
import sys
import subprocess
from pathlib import Path

# This notebook uses PyTorch only. Disable TensorFlow discovery in Transformers;
# this avoids the Keras 3 / tf-keras import path entirely.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Locate the repository root and install the Colab requirements when available.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements-colab.txt").exists():
        PROJECT_ROOT = candidate
        requirements_path = candidate / "requirements-colab.txt"
        break
    if (candidate / "requirements.txt").exists():
        PROJECT_ROOT = candidate
        requirements_path = candidate / "requirements.txt"
        break
else:
    raise FileNotFoundError("Could not find requirements-colab.txt or requirements.txt.")

# Ensure relative paths like tutorial_data/, mqnli_factual/, and mqnli_das/
# are rooted at the repository, even if Jupyter opens this file from notebooks/.
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print(f"Installing requirements from: {requirements_path}")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-r",
    str(requirements_path),
])

# If the packaged pyvene install is unavailable/broken, fall back to GitHub.
try:
    import pyvene  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "git+https://github.com/stanfordnlp/pyvene.git",
    ])

print("Setup done. Restart the kernel/runtime, then continue with the import cells below.")

Installing requirements from: /Users/abhiram3001/Downloads/course project stuff/requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 MB 23.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 20.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.8
    Uninstalling protobuf-4.25.8:
      Successfully uninstalled protobuf-4.25.8
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
  Attempting uninstall: keras
    Found existing installation: keras 3.11.3
    Uninstalling keras-3.11.3:
      Successfully uninstalled keras-3.11.3
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.16.1
    Uninstalli

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.34.1 which is incompatible.


: 

In [ ]:
import os

# PyTorch-only notebook: keep Transformers from importing TensorFlow/Keras.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

%load_ext autoreload
%autoreload 2
import pyvene

[autoreload of google.protobuf.descriptor failed: Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/anaconda3/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/opt/anaconda3/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/anaconda3/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 365, in update_class
    update_instances(old, new)
  File "/opt/anaconda3/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 323, in update_instances
    object.__setattr__(ref, "__class__", new)
TypeError: can't apply this __setattr__ to DescriptorMetaclass object
]
[autoreload of tensorflow.core.framework.tensor_shape_pb2 failed: Traceback (most recent call last):
  F

In [ ]:
import os
import json 
import random
import copy
import itertools
import numpy as np
from tqdm import tqdm, trange
from collections import Counter

import torch
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from transformers import TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score

from pyvene import CausalModel
# from pyvene.models.configuration_intervenable_model import RepresentationConfig
from pyvene import (
    IntervenableModel,
    RotatedSpaceIntervention,
    RepresentationConfig,
    IntervenableConfig
)

from pyvene.models.gpt2.modelings_intervenable_gpt2 import create_gpt2_lm
# from pyvene.models.gpt_neox.modelings_intervenable_gpt_neox import create_gpt_neox

### Output Directories: What Are `mqnli_factual` and `mqnli_das`?

These are **generated checkpoint/output directories**, not external datasets you need to download.

- `TRAIN_DIR = "mqnli_factual"` is where the notebook saves the factual GPT-2 model after the factual MQNLI training section (`trainer.save_model(TRAIN_DIR)`). Later cells reload this model with `create_gpt2_lm(name=TRAIN_DIR)`.
- `DAS_DIR = "mqnli_das"` is where pyvene saves the trained DAS `IntervenableModel` (`intervenable.save(DAS_DIR)`). Later cells reload it with `IntervenableModel.load(DAS_DIR, model)`.

So the intended order is:

1. Run factual training first; this creates `mqnli_factual/`.
2. Run DAS training after loading that factual model; this creates `mqnli_das/`.
3. Run evaluation/reload cells only after the corresponding directories exist.

The small MQNLI symbolic-signature JSON files are different: those live under `tutorial_data/` and are downloaded by the setup/data cell below if missing.

In [ ]:
TRAIN_DIR = 'mqnli_factual'
DAS_DIR = 'mqnli_das'

In [ ]:
seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

In [ ]:
# Ensure MQNLI signature JSON files are present.
# If you cloned pyvene/tutorials locally, these files may already exist.
# If not, this downloads the tiny tutorial_data JSONs from the pyvene GitHub repo.
from pathlib import Path
import urllib.request

tutorial_data_dir = Path("tutorial_data")
tutorial_data_dir.mkdir(exist_ok=True)

PYVENE_TUTORIAL_DATA_BASE = (
    "https://raw.githubusercontent.com/stanfordnlp/pyvene/main/"
    "tutorials/advanced_tutorials/tutorial_data"
)

required_mqnli_jsons = [
    "mqnli_q_projectivity.json",
    "mqnli_neg_signature.json",
    "mqnli_empty_signature.json",
    "mqnli_cont_signature.json",
    "mqnli_neg_cont_signature.json",
]

for fname in required_mqnli_jsons:
    fpath = tutorial_data_dir / fname
    if not fpath.exists():
        url = f"{PYVENE_TUTORIAL_DATA_BASE}/{fname}"
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(url, fpath)
print("MQNLI tutorial_data ready:", tutorial_data_dir.resolve())

## The MQNLI Task

Multiply-quantified natural language inference (MQNLI) is a variant of natural language inference (NLI) with a highly structured composition. Each datapoint is a pair of sentences, and each label is the logical relation between them (e.g., entailment, reverse entailment, no relation). Crucially, the logical relation can be computed compositionally: first compute the relation between the two sentences' nouns, then their determiners, and then combine these intermediate computations to find the relation between the sentences' noun phrases (NPs). 

In this section, we walk through the high-level causal structure of MQNLI with examples from the MQNLI dataset. We encourage first-time readers to read the original paper, [Geiger, Cases, Karttunen, and Potts (2018)](https://arxiv.org/pdf/1810.13033.pdf), before getting started.

In [ ]:
# JSON files generated from adapting MQNLI codebase
# https://github.com/atticusg/MultiplyQuantifiedData

class Hashabledict(dict):
    def __hash__(self):
        return hash(frozenset(self))

with open('tutorial_data/mqnli_q_projectivity.json') as f:
    determiner_signatures = json.load(f)
    determiner_signatures = Hashabledict({
        q1: Hashabledict({
            q2: Hashabledict({
                r1: Hashabledict({
                    r2: determiner_signatures[q1][q2][r1][r2]  
                    for r2 in determiner_signatures[q1][q2][r1]
                })  
                for r1 in determiner_signatures[q1][q2]
            })
            for q2 in determiner_signatures[q1]
        })
        for q1 in determiner_signatures
    })

with open('tutorial_data/mqnli_neg_signature.json') as f:
    negation_signature = Hashabledict(json.load(f))

with open('tutorial_data/mqnli_empty_signature.json') as f:
    emptystring_signature = Hashabledict(json.load(f))

with open('tutorial_data/mqnli_cont_signature.json') as f:
    compose_contradiction_signature = Hashabledict(json.load(f))

with open('tutorial_data/mqnli_neg_cont_signature.json') as f:
    compose_neg_contradiction_signature = Hashabledict(json.load(f))

In [ ]:
parents = {
    "N_P_O": [],
    "N_H_O": [],
    "N_O": ["N_P_O", "N_H_O"],
    "Adj_P_O": [],
    "Adj_H_O": [],
    "Adj_O": ["Adj_P_O", "Adj_H_O"],
    "NP_O": ["Adj_O", "N_O"],
    "Q_P_O": [],
    "Q_H_O": [],
    "Q_O": ["Q_P_O", "Q_H_O"],
    "V_P": [],
    "V_H": [],
    "V": ["V_P", "V_H"],
    "Adv_P": [],
    "Adv_H": [],
    "Adv": ["Adv_P", "Adv_H"],
    "VP": ["Adv", "V"],
    "QP_O": ["Q_O", "NP_O", "VP"],
    "Neg_P": [],
    "Neg_H": [],
    "Neg": ["Neg_P", "Neg_H"],
    "NegP": ["Neg", "QP_O"],
    "N_P_S": [],
    "N_H_S": [],
    "N_S": ["N_P_S", "N_H_S"],
    "Adj_P_S": [],
    "Adj_H_S": [],
    "Adj_S": ["Adj_P_S", "Adj_H_S"],
    "NP_S": ["Adj_S", "N_S"],
    "Q_P_S": [],
    "Q_H_S": [],
    "Q_S": ["Q_P_S", "Q_H_S"],
    "QP_S": ["Q_S", "NP_S", "NegP"]
}

In [ ]:
EMPTY = ""
IND = "independence"
EQV = "equivalence"
ENT = "entails"
REV = "reverse entails"
CON = "contradiction"
ALT = "alternation"
COV = "cover"
# all possible relation values from the original paper 
# (https://arxiv.org/pdf/1810.13033.pdf)
RELATIONS = [IND, EQV, ENT, REV, CON, ALT, COV]

Q_VALUES = [
    determiner_signatures[q1][q2]
    for q1 in determiner_signatures for q2 in determiner_signatures[q1]
]

values = {
    "N_P_O": ["tree", "rock"],
    "N_H_O": ["tree", "rock"],
    "N_O": [EQV, IND],
    "Adj_P_O": ["happy", "sad", EMPTY],
    "Adj_H_O": ["happy", "sad", EMPTY],
    "Adj_O": [EQV, IND, ENT, REV],
    "NP_O": [EQV, IND, ENT, REV],
    "Q_P_O": ["some", "every"],
    "Q_H_O": ["some", "every"],
    "Q_O": Q_VALUES,
    "V_P": ["climbed", "threw"],
    "V_H": ["climbed", "threw"],
    "V": [EQV, IND],
    "Adv_P": ["energetically", "joyfully", EMPTY],
    "Adv_H": ["energetically", "joyfully", EMPTY],
    "Adv": [EQV, IND, ENT, REV],
    "VP": [EQV, IND, ENT, REV],
    "QP_O": [EQV, IND, ENT, REV],
    "Neg_P": ["not", EMPTY],
    "Neg_H": ["not", EMPTY],
    "Neg": [
        negation_signature, 
        emptystring_signature, 
        compose_contradiction_signature, 
        compose_neg_contradiction_signature
    ],
    "NegP": RELATIONS,
    "N_P_S": ["child", "dog"],
    "N_H_S": ["child", "dog"],
    "N_S": [EQV, IND],
    "Adj_P_S": ["little", "cute", EMPTY],
    "Adj_H_S": ["little", "cute", EMPTY],
    "Adj_S": [EQV, IND, ENT, REV],
    "NP_S": [EQV, IND, ENT, REV],
    "Q_P_S": ["some", "every"],
    "Q_H_S": ["some", "every"],
    "Q_S": Q_VALUES,
    "QP_S": RELATIONS
}

In [ ]:
# adapted from original code for MQNLI:
# https://github.com/atticusg/MultiplyQuantifiedData/blob/master/natural_logic_model.py

def adj_merge(adj_p, adj_h):
    if adj_p == adj_h:
        return EQV
    if adj_p == EMPTY:
        return REV
    if adj_h == EMPTY:
        return ENT
    return IND

adv_merge = adj_merge

def noun_phrase(adj, noun):
    #merges a noun relation with an adjective relation
    #or a verb relation with an adverb relation
    # makes sense: if the objects are the same, then adjective's relation holds
    # otherwise, they're independent
    if noun == EQV:
        return adj
    return IND

verb_phrase = noun_phrase

def negation_merge(neg_p, neg_h):
    #merges negation
    if neg_p == neg_h and neg_p == EMPTY:
        return Hashabledict(emptystring_signature)
    if neg_p == neg_h and neg_p != EMPTY:
        return Hashabledict(negation_signature)
    if neg_p == EMPTY:
        return Hashabledict(compose_contradiction_signature)
    if neg_p != EMPTY:
        return Hashabledict(compose_neg_contradiction_signature)

negation_phrase = lambda neg, qp: neg[qp]

noun_merge = lambda n_p, n_h: EQV if n_p == n_h else IND
verb_merge = noun_merge

quantifier_merge = lambda q_p, q_h: determiner_signatures[q_p][q_h]

quantifier_phrase = lambda q, np, vp: q[np][vp]

functions = {
    "N_P_O": lambda: "tree",
    "N_H_O": lambda: "tree",
    "N_O": noun_merge,
    "Adj_P_O": lambda: "happy",
    "Adj_H_O": lambda: "happy",
    "Adj_O": adj_merge,
    "NP_O": noun_phrase,
    "Q_P_O": lambda: "some",
    "Q_H_O": lambda: "some",
    "Q_O": quantifier_merge,
    "V_P": lambda: "climbed",
    "V_H": lambda: "climbed",
    "V": verb_merge,
    "Adv_P": lambda: "energetically",
    "Adv_H": lambda: "energetically",
    "Adv": adv_merge,
    "VP": verb_phrase,
    "QP_O": quantifier_phrase,
    "Neg_P": lambda: "not",
    "Neg_H": lambda: "not",
    "Neg": negation_merge,
    "NegP": negation_phrase,
    "N_P_S": lambda: "dog",
    "N_H_S": lambda: "dog",
    "N_S": noun_merge,
    "Adj_P_S": lambda: "cute",
    "Adj_H_S": lambda: "cute",
    "Adj_S": adj_merge,
    "NP_S": noun_phrase,
    "Q_P_S": lambda: "some",
    "Q_H_S": lambda: "some",
    "Q_S": quantifier_merge,
    "QP_S": quantifier_phrase
}

In [ ]:
# coordinates to display the MQNLI tree
pos = {
    "N_P_O": (32, 0.3),
    "N_H_O": (30, 0.7),
    "N_O": (31, 1.3),
    "Adj_P_O": (28, 0),
    "Adj_H_O": (26, 0.5),
    "Adj_O": (27, 1),
    "NP_O": (29, 2),
    "Q_P_O": (24, 1.3),
    "Q_H_O": (22, 1.7),
    "Q_O": (23, 2.5),
    "V_P": (21, 0),
    "V_H": (19, 0.5),
    "V": (20, 1),
    "Adv_P": (17, -0.3),
    "Adv_H": (15, 0.2),
    "Adv": (16, 0.7),
    "VP": (18, 2),
    "QP_O": (25, 3),
    "Neg_P": (13, 2.5),
    "Neg_H": (11, 3),
    "Neg": (12, 3.5),
    "NegP": (14, 4),
    "N_P_S": (9, 2.2),
    "N_H_S": (7, 2.8),
    "N_S": (8, 3.3),
    "Adj_P_S": (5, 1.5),
    "Adj_H_S": (3, 2),
    "Adj_S": (4, 2.5),
    "NP_S": (6, 4.3),
    "Q_P_S": (2, 3.2),
    "Q_H_S": (0, 3.5),
    "Q_S": (1, 4),
    "QP_S": (10, 5)
}

In [ ]:
variables = list(parents.keys())  # pretty sure this preserves order?

In [ ]:
mqlni_model = CausalModel(variables, values, parents, functions, pos=pos)

We begin by visualizing the high-level structure of the MQNLI task. All leaf nodes consist of text entries (e.g., Adj_P_O might be "fun", N_H_O might be "child"), with one copy corresponding to the premise sentence (P) and the other corresponding to the hypothesis (H). All other nodes take on relation values (e.g., entailment, reverse entailment, no relation), or mappings between relations (e.g., (entailment, entailment) --> no entailment).

In [ ]:
mqlni_model.print_structure()

### Example MQNLI Input

Here we walk through an example MQNLI input, presented below.
```json
Premise: Every dog climbed some tree.
Hypothesis: Some dog did not climb every tree.
```

The output for this sentence pair is `contradiction`, because the premise entails that there cannot be a dog who climbed no tree. 

In [ ]:
inputs = {
    # premise
    "Q_P_S": "every",
    "Adj_P_S": EMPTY,
    "N_P_S": "dog",
    "Neg_P": EMPTY,
    "Adv_P": EMPTY,
    "V_P": "climbed",
    "Q_P_O": "some",
    "Adj_P_O": EMPTY,
    "N_P_O": "tree",
    # hypothesis
    "Q_H_S": "some",
    "Adj_H_S": EMPTY,
    "N_H_S": "dog",
    "Neg_H": "not",
    "Adv_H": EMPTY,
    "V_H": "climbed",
    "Q_H_O": "some",
    "Adj_H_O": EMPTY,
    "N_H_O": "tree"
}

setting = mqlni_model.run_forward(inputs)

In [ ]:
def print_premise(setting):
    print(
        setting["Q_P_S"],
        setting["Adj_P_S"],
        setting["N_P_S"],
        setting["Neg_P"],
        setting["Adv_P"],
        setting["V_P"],
        setting["Q_P_O"],
        setting["Adj_P_O"],
        setting["N_P_O"]
    )

def print_hypothesis(setting):
    print(
        setting["Q_H_S"],
        setting["Adj_H_S"],
        setting["N_H_S"],
        setting["Neg_H"],
        setting["Adv_H"],
        setting["V_H"],
        setting["Q_H_O"],
        setting["Adj_H_O"],
        setting["N_H_O"]
    )

print_premise(setting)
print_hypothesis(setting)

print(setting["QP_S"]) # the output is in the root node, QP_S

We display the structure of the MQNLI example below. Note that the intermediate relation values compose with each other to produce the final output. For instance, since `N_O` and `Adj_O` both take on the value `equivalence`, then `NP_O` is also an equivalence relation.

In [ ]:
display = {v: not isinstance(values[v][0], dict) for v in variables}
mqlni_model.print_setting(setting, display=display)

### MQNLI with an Intervention

Below is a run of the MQNLI logic model where we intervene on the object quantifier (`QP_O`) and swap its relation value from `equivalence` to `contradiction`. Note that this changes the final output from `contradiction` to `entailment`.

In [ ]:
intervention = {**inputs, 'QP_O': 'contradiction'}
setting = mqlni_model.run_forward(intervention)
print_premise(setting)
print_hypothesis(setting)
print(setting["QP_S"])

mqlni_model.print_setting(setting, display=display)

## Project-ready factual training + DAS experiment suite

The original tutorial used a tiny factual training run and one DAS configuration.  
This section turns the notebook into a reusable experiment harness for the CS221M project:

1. train/check a factual MQNLI model;
2. run DAS on chosen high-level causal variables;
3. compare final-label, intermediate-relation, lexical-identity, lexical-relation, and monotonicity variables;
4. run layer, subspace-dimension, and token-position sweeps;
5. include simple controls: majority-label baseline, untrained/random-rotation baseline, shuffled-label control, and full-stream interchange baseline.

The defaults are intentionally small for a smoke test. For real runs, flip `RUN_SMOKE_TEST = False` and/or edit the config block.

In [ ]:
# Extra imports for experiments/results.
import math
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Make notebook runs quieter/reproducible.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

RUN_SMOKE_TEST = True  # True = quick syntax/debug run; False = project-scale defaults.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

PROJECT_TRAIN_DIR = "mqnli_factual_project"
PROJECT_DAS_ROOT = "mqnli_das_experiments"
RESULTS_DIR = "mqnli_results"
os.makedirs(PROJECT_DAS_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if RUN_SMOKE_TEST:
    EXP = dict(
        factual_train_n=300,
        factual_test_n=100,
        factual_epochs=1,
        factual_lr=5e-5,
        factual_batch_size=8,
        das_train_n=100,
        das_test_n=100,
        das_epochs=1,
        das_lr=1e-3,
        das_batch_size=8,
        das_subspace_dim=32,
        das_layer=10,
        max_length=64,
        factual_accuracy_gate=0.80,
    )
else:
    EXP = dict(
        factual_train_n=5000,
        factual_test_n=1000,
        factual_epochs=5,
        factual_lr=5e-5,
        factual_batch_size=16,
        das_train_n=1000,
        das_test_n=1000,
        das_epochs=3,
        das_lr=1e-3,
        das_batch_size=16,
        das_subspace_dim=128,
        das_layer=10,
        max_length=64,
        factual_accuracy_gate=0.90,
    )

EXP

### Text formatting and tokenization helpers

The LM is trained as a causal language model. We give it a prompt ending in `Relation:` and put the target relation token at the final sequence position. During evaluation, the prediction at position `MAX_LENGTH - 2` predicts the label stored at `MAX_LENGTH - 1`.

In [ ]:
IGNORE_INDEX = -100
MAX_LENGTH = EXP["max_length"]

def clean_join(tokens):
    # Join phrase tokens while removing empty-string modifiers.
    return " ".join(str(t) for t in tokens if str(t).strip() != "")

def premise_to_string(setting):
    return clean_join([
        setting["Q_P_S"],
        setting["Adj_P_S"],
        setting["N_P_S"],
        setting["Neg_P"],
        setting["Adv_P"],
        setting["V_P"],
        setting["Q_P_O"],
        setting["Adj_P_O"],
        setting["N_P_O"],
    ])

def hypothesis_to_string(setting):
    return clean_join([
        setting["Q_H_S"],
        setting["Adj_H_S"],
        setting["N_H_S"],
        setting["Neg_H"],
        setting["Adv_H"],
        setting["V_H"],
        setting["Q_H_O"],
        setting["Adj_H_O"],
        setting["N_H_O"],
    ])

def preprocess_input(setting):
    return f"Premise: {premise_to_string(setting)}\nHypothesis: {hypothesis_to_string(setting)}\nRelation: "

def preprocess_output(setting, output_var="QP_S"):
    return setting[output_var]

def set_tokenizer_padding(tokenizer):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def tokenize_texts(tokenizer, texts, max_length=MAX_LENGTH):
    return tokenizer(
        texts,
        padding="max_length",
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )

def first_label_token_id(tokenizer, label):
    # The original tutorial uses the first token of each relation label.
    return tokenizer(str(label), add_special_tokens=False)["input_ids"][0]

def relation_token_map(tokenizer):
    mapping = {label: first_label_token_id(tokenizer, label) for label in RELATIONS}
    inv = {v: k for k, v in mapping.items()}
    if len(inv) != len(mapping):
        print("WARNING: some relation labels share a first token. First-token classification is ambiguous.")
        print(mapping)
    return mapping, inv

def preprocess_factual(X, y, tokenizer, max_length=MAX_LENGTH, output_var="QP_S"):
    examples = [preprocess_input(x) for x in X]
    label_ids = [first_label_token_id(tokenizer, preprocess_output(yy, output_var)) for yy in y]
    enc = tokenize_texts(tokenizer, examples, max_length=max_length)
    enc["labels"] = torch.full_like(enc["input_ids"], IGNORE_INDEX)
    enc["labels"][:, -1] = torch.tensor(label_ids, dtype=torch.long)
    return enc

def generate_factual_splits(train_n=None, test_n=None, seed=42):
    train_n = train_n or EXP["factual_train_n"]
    test_n = test_n or EXP["factual_test_n"]

    random.seed(seed)
    np.random.seed(seed)

    train_examples = mqlni_model.generate_factual_dataset(
        train_n,
        sampler=mqlni_model.sample_input_tree_balanced,
        return_tensors=False,
    )
    test_examples = mqlni_model.generate_factual_dataset(
        test_n,
        sampler=mqlni_model.sample_input_tree_balanced,
        return_tensors=False,
    )

    X_train = [e["input_ids"] for e in train_examples]
    y_train = [e["labels"] for e in train_examples]
    X_test = [e["input_ids"] for e in test_examples]
    y_test = [e["labels"] for e in test_examples]

    return X_train, y_train, X_test, y_test

def label_counter(y, output_var="QP_S"):
    return Counter([yy[output_var] for yy in y])

### Factual model training gate

DAS only becomes meaningful after the factual model has learned MQNLI. The important gate is factual accuracy. For the project-scale run, aim for at least 90% test accuracy before interpreting DAS.

In [ ]:
def make_accuracy_metric(tokenizer):
    token_to_relation = relation_token_map(tokenizer)[1]

    def accuracy_metric(eval_pred):
        labels = eval_pred.label_ids[:, -1]
        # For GPT-2 causal LM, logits at sequence position -2 predict token at -1.
        predictions = eval_pred.predictions.argmax(axis=-1)[:, -2]
        return {"accuracy": accuracy_score(y_true=labels, y_pred=predictions)}
    return accuracy_metric

def evaluate_factual_predictions(trainer, ds, tokenizer, split_name="eval"):
    pred = trainer.predict(ds)
    labels = pred.label_ids[:, -1]
    preds = pred.predictions.argmax(axis=-1)[:, -2]

    token_to_relation = relation_token_map(tokenizer)[1]
    df = pd.DataFrame({
        "true_token": labels,
        "pred_token": preds,
        "true_relation": [token_to_relation.get(int(t), f"tok_{int(t)}") for t in labels],
        "pred_relation": [token_to_relation.get(int(t), f"tok_{int(t)}") for t in preds],
    })
    acc = accuracy_score(df["true_token"], df["pred_token"])
    print(f"{split_name} factual accuracy: {acc:.4f}")

    try:
        print(classification_report(df["true_relation"], df["pred_relation"], zero_division=0))
    except Exception as e:
        print("classification_report failed:", e)

    return {"accuracy": acc, "predictions": df}

def make_training_args(**kwargs):
    # Some transformers versions renamed evaluation_strategy/save_strategy.
    try:
        return TrainingArguments(**kwargs)
    except TypeError:
        kwargs = dict(kwargs)
        if "evaluation_strategy" in kwargs:
            kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
        if "save_strategy" in kwargs:
            kwargs["save_strategy"] = kwargs.pop("save_strategy")
        return TrainingArguments(**kwargs)

def train_factual_model(
    train_n=None,
    test_n=None,
    train_dir=PROJECT_TRAIN_DIR,
    epochs=None,
    lr=None,
    batch_size=None,
    seed=42,
):
    train_n = train_n or EXP["factual_train_n"]
    test_n = test_n or EXP["factual_test_n"]
    epochs = epochs or EXP["factual_epochs"]
    lr = lr or EXP["factual_lr"]
    batch_size = batch_size or EXP["factual_batch_size"]

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    config, tokenizer, model = create_gpt2_lm()
    tokenizer = set_tokenizer_padding(tokenizer)

    print("Label-token map:")
    display(pd.DataFrame(relation_token_map(tokenizer)[0].items(), columns=["relation", "first_token_id"]))

    X_train, y_train, X_test, y_test = generate_factual_splits(train_n, test_n, seed=seed)
    print("Train label distribution:", label_counter(y_train))
    print("Test label distribution:", label_counter(y_test))

    train_dataset = preprocess_factual(X_train, y_train, tokenizer)
    test_dataset = preprocess_factual(X_test, y_test, tokenizer)

    train_ds = Dataset.from_dict(train_dataset)
    test_ds = Dataset.from_dict(test_dataset)

    args = make_training_args(
        output_dir=train_dir,
        overwrite_output_dir=True,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        report_to="none",
        use_cpu=(DEVICE == "cpu"),
        logging_steps=25,
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=make_accuracy_metric(tokenizer),
    )

    trainer.train()
    trainer.save_model(train_dir)

    train_eval = evaluate_factual_predictions(trainer, train_ds, tokenizer, "train")
    test_eval = evaluate_factual_predictions(trainer, test_ds, tokenizer, "test")

    if test_eval["accuracy"] < EXP["factual_accuracy_gate"]:
        print(
            f"WARNING: factual test accuracy {test_eval['accuracy']:.3f} is below gate "
            f"{EXP['factual_accuracy_gate']:.3f}. DAS results should be treated as debugging only."
        )

    return {
        "config": config,
        "tokenizer": tokenizer,
        "model": model,
        "trainer": trainer,
        "train_ds": train_ds,
        "test_ds": test_ds,
        "train_eval": train_eval,
        "test_eval": test_eval,
        "train_dir": train_dir,
    }

def load_factual_model(train_dir=PROJECT_TRAIN_DIR):
    config, tokenizer, model = create_gpt2_lm(name=train_dir)
    tokenizer = set_tokenizer_padding(tokenizer)
    return config, tokenizer, model

In [ ]:
# Run this cell to train the factual model.
# For a quick smoke test this may still have low accuracy; project-scale settings should pass the gate.
factual_run = train_factual_model()
tokenizer = factual_run["tokenizer"]

### Counterfactual data helpers

For a chosen high-level variable `V`, pyvene's causal model builds base/source pairs and computes the symbolic counterfactual output. DAS is then trained so that swapping a low-dimensional neural subspace from source into base makes the LM predict the symbolic counterfactual label.

In [ ]:
def sample_variable_intervention(variable_name):
    # Return an intervention sampler for one high-level variable.
    possible_values = list(mqlni_model.values[variable_name])

    def _sampler(*args, **kwargs):
        return {variable_name: random.choice(possible_values)}

    return _sampler

def fixed_intervention_id(*args, **kwargs):
    return 0

def generate_counterfactual_data(
    variable_name,
    n=None,
    source_batch_size=1,
    seed=42,
):
    n = n or EXP["das_train_n"]
    random.seed(seed)
    np.random.seed(seed)

    return mqlni_model.generate_counterfactual_dataset(
        n,
        fixed_intervention_id,
        source_batch_size,
        sampler=mqlni_model.sample_input_tree_balanced,
        intervention_sampler=sample_variable_intervention(variable_name),
        return_tensors=False,
    )

def summarize_counterfactual_data(data, variable_name):
    base_counts = Counter([d["base_labels"]["QP_S"] for d in data])
    cf_counts = Counter([d["labels"]["QP_S"] for d in data])
    rows = []
    for rel in RELATIONS:
        rows.append({
            "relation": rel,
            "base_count": base_counts.get(rel, 0),
            "counterfactual_count": cf_counts.get(rel, 0),
        })
    print(f"Counterfactual dataset for variable: {variable_name}")
    display(pd.DataFrame(rows))
    majority = max(cf_counts.values()) / max(1, sum(cf_counts.values()))
    print(f"Majority-label baseline on counterfactual labels: {majority:.4f}")
    return {"base_counts": base_counts, "counterfactual_counts": cf_counts, "majority_baseline": majority}

def preprocess_counterfactual(data, tokenizer, max_length=MAX_LENGTH):
    preprocessed_data = []
    for d in data:
        base_text = preprocess_input(d["input_ids"])
        source_texts = [preprocess_input(d["source_input_ids"][0])]

        label_id = first_label_token_id(tokenizer, d["labels"]["QP_S"])
        base_label_id = first_label_token_id(tokenizer, d["base_labels"]["QP_S"])

        preprocessed = {}
        preprocessed["input"] = tokenize_texts(tokenizer, base_text, max_length=max_length)
        preprocessed["source"] = [tokenize_texts(tokenizer, source_texts[0], max_length=max_length)]

        preprocessed["label"] = torch.full_like(preprocessed["input"]["input_ids"], IGNORE_INDEX)
        preprocessed["label"][:, -1] = label_id

        preprocessed["base_label"] = torch.full_like(preprocessed["input"]["input_ids"], IGNORE_INDEX)
        preprocessed["base_label"][:, -1] = base_label_id

        preprocessed["intervention_id"] = torch.tensor(d["intervention_id"])
        preprocessed_data.append(preprocessed)

    return preprocessed_data

def shuffle_counterfactual_labels(preprocessed_data, seed=0):
    # Control: train on the same base/source pairs but shuffled counterfactual labels.
    rng = random.Random(seed)
    shuffled = copy.deepcopy(preprocessed_data)
    labels = [item["label"].clone() for item in shuffled]
    rng.shuffle(labels)
    for item, label in zip(shuffled, labels):
        item["label"] = label
    return shuffled

def majority_counterfactual_accuracy(raw_data):
    counts = Counter([d["labels"]["QP_S"] for d in raw_data])
    return max(counts.values()) / max(1, sum(counts.values()))

### DAS core functions

`train_single_das` is the main reusable primitive. It trains one rotated subspace for one variable at one layer/token/dimension, evaluates on held-out counterfactuals, and records useful baselines.

In [ ]:
def get_hidden_size(model):
    for attr in ["n_embd", "hidden_size", "d_model"]:
        if hasattr(model.config, attr):
            return int(getattr(model.config, attr))
    raise ValueError("Could not infer hidden size from model.config.")

def make_intervenable_model(
    model,
    layer,
    subspace_dim,
    representation_type="block_output",
    use_fast=True,
):
    config = IntervenableConfig(
        model_type=type(model),
        representations=[
            RepresentationConfig(
                layer,
                representation_type,
                "pos",
                1,
                subspace_partition=[[0, subspace_dim]],
            )
        ],
        intervention_types=RotatedSpaceIntervention,
    )
    intervenable = IntervenableModel(config, model, use_fast=use_fast)
    if DEVICE == "cuda":
        intervenable.set_device("cuda")
    intervenable.disable_model_gradients()
    return intervenable

def intervention_location(batch_size, position=None):
    # Default: answer-prediction position.
    if position is None:
        position = MAX_LENGTH - 2
    return {"sources->base": ([[[position]] * batch_size], [[[position]] * batch_size])}

def das_compute_loss(outputs, labels):
    # Standard causal-LM shift loss; labels are -100 except final position.
    shift_logits = outputs[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss_fct = CrossEntropyLoss()
    return loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

def das_accuracy_from_logits(logits, labels):
    preds = logits.argmax(-1)
    return accuracy_score(
        y_pred=preds[..., -2].squeeze().clone().detach().cpu().numpy(),
        y_true=labels[..., -1].squeeze().clone().detach().cpu().numpy(),
    )

def optimizer_for_intervenable(intervenable, lr=1e-3):
    params = []
    for _, intervention in intervenable.interventions.items():
        if hasattr(intervention, "rotate_layer"):
            params.extend(list(intervention.rotate_layer.parameters()))
        else:
            params.extend(list(intervention.parameters()))
    if not params:
        raise ValueError("No trainable intervention parameters found.")
    return torch.optim.Adam(params, lr=lr)

def evaluate_intervenable(
    intervenable,
    dataset,
    batch_size=None,
    position=None,
    return_predictions=False,
):
    batch_size = batch_size or EXP["das_batch_size"]
    intervenable.model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(DataLoader(dataset, batch_size=batch_size, drop_last=False), desc="DAS eval"):
            effective_bs = batch["input"]["input_ids"].shape[0]
            inputs = {k: v.to(intervenable.get_device()) for k, v in batch["input"].items()}
            sources = [{k: v.to(intervenable.get_device()) for k, v in s.items()} for s in batch["source"]]

            _, cf_outputs = intervenable(
                inputs,
                sources,
                intervention_location(effective_bs, position=position),
                subspaces=[[[0]] * effective_bs],
            )

            logits = cf_outputs.logits
            labels = batch["label"].to(intervenable.get_device())

            preds = logits.argmax(-1)[..., -2].detach().cpu()
            truth = labels[..., -1].detach().cpu()
            all_preds.append(preds)
            all_labels.append(truth)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)

    out = {"accuracy": acc}
    if return_predictions:
        out["pred_tokens"] = all_preds
        out["label_tokens"] = all_labels
    return out

def train_intervenable_on_dataset(
    intervenable,
    train_dataset,
    epochs=None,
    lr=None,
    batch_size=None,
    position=None,
    verbose=True,
):
    epochs = epochs or EXP["das_epochs"]
    lr = lr or EXP["das_lr"]
    batch_size = batch_size or EXP["das_batch_size"]

    optimizer = optimizer_for_intervenable(intervenable, lr=lr)

    if verbose:
        print("Intervention trainable parameters:", intervenable.count_parameters())

    total_step = 0
    for epoch in range(int(epochs)):
        intervenable.model.train()
        iterator = tqdm(
            DataLoader(train_dataset, batch_size=batch_size, drop_last=True),
            desc=f"DAS epoch {epoch}",
        )
        for batch in iterator:
            effective_bs = batch["input"]["input_ids"].shape[0]
            inputs = {k: v.to(intervenable.get_device()) for k, v in batch["input"].items()}
            sources = [{k: v.to(intervenable.get_device()) for k, v in s.items()} for s in batch["source"]]

            _, cf_outputs = intervenable(
                inputs,
                sources,
                intervention_location(effective_bs, position=position),
                subspaces=[[[0]] * effective_bs],
            )

            labels = batch["label"].to(intervenable.get_device())
            loss = das_compute_loss(cf_outputs.logits, labels)
            acc = das_accuracy_from_logits(cf_outputs.logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if hasattr(intervenable, "set_zero_grad"):
                intervenable.set_zero_grad()

            total_step += 1
            iterator.set_postfix({"loss": float(loss.detach().cpu()), "acc": acc})

    return intervenable

def train_single_das(
    variable_name="NegP",
    layer=None,
    subspace_dim=None,
    position=None,
    train_n=None,
    test_n=None,
    train_dir=PROJECT_TRAIN_DIR,
    das_root=PROJECT_DAS_ROOT,
    epochs=None,
    lr=None,
    batch_size=None,
    seed=42,
    shuffle_train_labels=False,
    save=True,
):
    layer = EXP["das_layer"] if layer is None else layer
    subspace_dim = EXP["das_subspace_dim"] if subspace_dim is None else subspace_dim
    train_n = train_n or EXP["das_train_n"]
    test_n = test_n or EXP["das_test_n"]
    epochs = epochs or EXP["das_epochs"]
    lr = lr or EXP["das_lr"]
    batch_size = batch_size or EXP["das_batch_size"]

    _, tokenizer, base_model = load_factual_model(train_dir)
    tokenizer = set_tokenizer_padding(tokenizer)

    train_raw = generate_counterfactual_data(variable_name, n=train_n, seed=seed)
    test_raw = generate_counterfactual_data(variable_name, n=test_n, seed=seed + 1)

    _ = summarize_counterfactual_data(train_raw, variable_name)

    train_dataset = preprocess_counterfactual(train_raw, tokenizer)
    test_dataset = preprocess_counterfactual(test_raw, tokenizer)

    if shuffle_train_labels:
        print("Using shuffled training labels control.")
        train_dataset = shuffle_counterfactual_labels(train_dataset, seed=seed)

    intervenable = make_intervenable_model(base_model, layer=layer, subspace_dim=subspace_dim)

    # Random/untrained rotated-subspace baseline.
    untrained_eval = evaluate_intervenable(intervenable, test_dataset, batch_size=batch_size, position=position)

    # DAS training.
    intervenable = train_intervenable_on_dataset(
        intervenable,
        train_dataset,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        position=position,
    )
    test_eval = evaluate_intervenable(intervenable, test_dataset, batch_size=batch_size, position=position)

    save_dir = None
    if save:
        safe_var = re.sub(r"[^A-Za-z0-9_]+", "_", variable_name)
        pos_name = "answer" if position is None else str(position)
        suffix = "shuffled" if shuffle_train_labels else "das"
        save_dir = os.path.join(das_root, f"{safe_var}_L{layer}_D{subspace_dim}_P{pos_name}_{suffix}")
        intervenable.save(save_dir)

    row = {
        "variable": variable_name,
        "layer": layer,
        "subspace_dim": subspace_dim,
        "position": MAX_LENGTH - 2 if position is None else position,
        "train_n": train_n,
        "test_n": test_n,
        "epochs": epochs,
        "lr": lr,
        "majority_baseline": majority_counterfactual_accuracy(test_raw),
        "untrained_rotation_accuracy": untrained_eval["accuracy"],
        "das_accuracy": test_eval["accuracy"],
        "shuffle_train_labels": shuffle_train_labels,
        "save_dir": save_dir,
    }
    print(pd.Series(row))
    return row, intervenable

### One-command smoke tests

Run these first. They test the main project questions in miniature:

- `QP_S`: final-label sanity check.
- `NegP`: the notebook's intermediate semantic variable.
- `N_P_O`, `N_H_O`, `N_O`: lexical identity versus lexical relation.

In [ ]:
VARIABLE_GROUPS = {
    "final_label": ["QP_S"],
    "notebook_target": ["NegP"],
    "lexical_identity": ["N_P_O", "N_H_O", "N_P_S", "N_H_S"],
    "lexical_relation": ["N_O", "N_S", "V", "Adj_O", "Adj_S"],
    "phrase_relation": ["NP_O", "NP_S", "VP", "QP_O", "NegP"],
    "monotonicity_context": ["Q_O", "Q_S"],
}

SMOKE_VARIABLES = ["QP_S", "NegP", "N_P_O", "N_H_O", "N_O"]

def run_variable_comparison(
    variables=None,
    layer=None,
    subspace_dim=None,
    position=None,
    seed=42,
    shuffle_control=False,
):
    variables = variables or SMOKE_VARIABLES
    rows = []
    for i, var in enumerate(variables):
        print("\n" + "=" * 80)
        print(f"Running DAS variable comparison: {var}")
        row, _ = train_single_das(
            variable_name=var,
            layer=layer,
            subspace_dim=subspace_dim,
            position=position,
            seed=seed + 10 * i,
            shuffle_train_labels=False,
        )
        rows.append(row)

        if shuffle_control:
            ctrl_row, _ = train_single_das(
                variable_name=var,
                layer=layer,
                subspace_dim=subspace_dim,
                position=position,
                seed=seed + 10 * i,
                shuffle_train_labels=True,
                save=False,
            )
            ctrl_row["control_type"] = "shuffled_labels"
            rows.append(ctrl_row)

    df = pd.DataFrame(rows)
    out_path = os.path.join(RESULTS_DIR, "variable_comparison.csv")
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)
    display(df)
    return df

# Run the smoke variable comparison.
# For project-scale results, first get strong factual accuracy, then use more train/test examples.
variable_results = run_variable_comparison(
    variables=SMOKE_VARIABLES if RUN_SMOKE_TEST else ["QP_S", "NegP", "N_P_O", "N_H_O", "N_O", "NP_O", "Q_S"],
    shuffle_control=False,
)

### Layer sweep

This asks where different high-level variables become DAS-alignable. A good final-project plot is DAS IIA/accuracy versus layer for several variables.

In [ ]:
def run_layer_sweep(
    variable_name="NegP",
    layers=None,
    subspace_dim=None,
    position=None,
    seed=123,
):
    if layers is None:
        layers = [8, 10] if RUN_SMOKE_TEST else [0, 2, 4, 6, 8, 10, 11]
    subspace_dim = subspace_dim or EXP["das_subspace_dim"]

    rows = []
    for i, layer in enumerate(layers):
        print("\n" + "=" * 80)
        print(f"Layer sweep: variable={variable_name}, layer={layer}")
        row, _ = train_single_das(
            variable_name=variable_name,
            layer=layer,
            subspace_dim=subspace_dim,
            position=position,
            seed=seed + i,
        )
        rows.append(row)

    df = pd.DataFrame(rows)
    out_path = os.path.join(RESULTS_DIR, f"layer_sweep_{variable_name}.csv")
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)
    display(df)
    return df

def plot_layer_sweep(df, title=None):
    plt.figure(figsize=(7, 4))
    for var, group in df.groupby("variable"):
        group = group.sort_values("layer")
        plt.plot(group["layer"], group["das_accuracy"], marker="o", label=var)
        plt.plot(group["layer"], group["majority_baseline"], linestyle="--", alpha=0.5)
    plt.xlabel("Layer")
    plt.ylabel("Held-out counterfactual accuracy")
    plt.title(title or "DAS layer sweep")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Uncomment for a layer sweep after the factual model is learned.
# layer_results = run_layer_sweep("NegP")
# plot_layer_sweep(layer_results, "NegP layer sweep")

### Subspace-dimension sweep

This asks how distributed/compressed the aligned variable is. Try small dimensions first, then 64/128/256.

In [ ]:
def run_dimension_sweep(
    variable_name="NegP",
    dims=None,
    layer=None,
    position=None,
    seed=456,
):
    if dims is None:
        dims = [8, 32] if RUN_SMOKE_TEST else [1, 4, 8, 16, 32, 64, 128, 256]
    layer = EXP["das_layer"] if layer is None else layer

    rows = []
    for i, dim in enumerate(dims):
        print("\n" + "=" * 80)
        print(f"Dimension sweep: variable={variable_name}, dim={dim}")
        row, _ = train_single_das(
            variable_name=variable_name,
            layer=layer,
            subspace_dim=dim,
            position=position,
            seed=seed + i,
        )
        rows.append(row)

    df = pd.DataFrame(rows)
    out_path = os.path.join(RESULTS_DIR, f"dimension_sweep_{variable_name}.csv")
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)
    display(df)
    return df

def plot_dimension_sweep(df, title=None):
    plt.figure(figsize=(7, 4))
    for var, group in df.groupby("variable"):
        group = group.sort_values("subspace_dim")
        plt.plot(group["subspace_dim"], group["das_accuracy"], marker="o", label=var)
        plt.plot(group["subspace_dim"], group["majority_baseline"], linestyle="--", alpha=0.5)
    plt.xscale("log", base=2)
    plt.xlabel("Subspace dimension")
    plt.ylabel("Held-out counterfactual accuracy")
    plt.title(title or "DAS subspace-dimension sweep")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Uncomment for a dimension sweep.
# dim_results = run_dimension_sweep("NegP")
# plot_dimension_sweep(dim_results, "NegP dimension sweep")

### Token-position inspection and position sweep

The tutorial intervenes at the answer-prediction position. For lexical variables, it is also useful to intervene near the relevant noun/verb/negation tokens. The helper below shows token positions for a concrete MQNLI example so you can choose positions for a sweep.

In [ ]:
def inspect_token_positions(setting=None, tokenizer=None):
    if tokenizer is None:
        _, tokenizer, _ = load_factual_model(PROJECT_TRAIN_DIR)
        tokenizer = set_tokenizer_padding(tokenizer)
    if setting is None:
        ex = mqlni_model.generate_factual_dataset(
            1,
            sampler=mqlni_model.sample_input_tree_balanced,
            return_tensors=False,
        )[0]
        setting = ex["input_ids"]

    text = preprocess_input(setting)
    enc = tokenizer(text, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    df = pd.DataFrame({"position": list(range(len(tokens))), "token": tokens})
    print(text)
    display(df)
    print("Default answer-prediction position used in this notebook:", MAX_LENGTH - 2)
    return df

def run_position_sweep(
    variable_name="N_O",
    positions=None,
    layer=None,
    subspace_dim=None,
    seed=789,
):
    if positions is None:
        # Replace these with positions discovered by inspect_token_positions.
        positions = [MAX_LENGTH - 2]
    layer = EXP["das_layer"] if layer is None else layer
    subspace_dim = subspace_dim or EXP["das_subspace_dim"]

    rows = []
    for i, pos in enumerate(positions):
        print("\n" + "=" * 80)
        print(f"Position sweep: variable={variable_name}, position={pos}")
        row, _ = train_single_das(
            variable_name=variable_name,
            layer=layer,
            subspace_dim=subspace_dim,
            position=pos,
            seed=seed + i,
        )
        rows.append(row)

    df = pd.DataFrame(rows)
    out_path = os.path.join(RESULTS_DIR, f"position_sweep_{variable_name}.csv")
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)
    display(df)
    return df

# Inspect tokenization before picking token positions.
# token_positions = inspect_token_positions()

# Then set positions manually, e.g. positions=[5, 12, 20, MAX_LENGTH - 2].
# pos_results = run_position_sweep("N_O", positions=[MAX_LENGTH - 2])

### Full-stream interchange baseline / activation-patching-style localization

This is a coarse baseline: instead of learning a small rotated subspace, swap the full residual stream at a layer and position. With full hidden dimension, the random rotation should not matter, so the untrained intervention acts like a full-stream interchange intervention at that site.

Use this as a quick localization baseline before expensive DAS sweeps.

In [ ]:
def full_stream_interchange_baseline(
    variable_name="NegP",
    layers=None,
    position=None,
    test_n=None,
    train_dir=PROJECT_TRAIN_DIR,
    seed=999,
):
    layers = layers or ([8, 10] if RUN_SMOKE_TEST else list(range(12)))
    test_n = test_n or EXP["das_test_n"]

    _, tokenizer, _ = load_factual_model(train_dir)
    test_raw = generate_counterfactual_data(variable_name, n=test_n, seed=seed)
    test_dataset = preprocess_counterfactual(test_raw, tokenizer)

    rows = []
    for layer in layers:
        print("\n" + "=" * 80)
        print(f"Full-stream interchange: variable={variable_name}, layer={layer}")
        _, _, base_model = load_factual_model(train_dir)
        hidden = get_hidden_size(base_model)
        intervenable = make_intervenable_model(base_model, layer=layer, subspace_dim=hidden)
        eval_result = evaluate_intervenable(
            intervenable,
            test_dataset,
            batch_size=EXP["das_batch_size"],
            position=position,
        )
        rows.append({
            "variable": variable_name,
            "layer": layer,
            "position": MAX_LENGTH - 2 if position is None else position,
            "subspace_dim": hidden,
            "full_stream_accuracy": eval_result["accuracy"],
            "majority_baseline": majority_counterfactual_accuracy(test_raw),
        })

    df = pd.DataFrame(rows)
    out_path = os.path.join(RESULTS_DIR, f"full_stream_interchange_{variable_name}.csv")
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)
    display(df)
    return df

def plot_full_stream_baseline(df, title=None):
    plt.figure(figsize=(7, 4))
    for var, group in df.groupby("variable"):
        group = group.sort_values("layer")
        plt.plot(group["layer"], group["full_stream_accuracy"], marker="o", label=var)
        plt.plot(group["layer"], group["majority_baseline"], linestyle="--", alpha=0.5)
    plt.xlabel("Layer")
    plt.ylabel("Full-stream counterfactual accuracy")
    plt.title(title or "Full-stream interchange baseline")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Uncomment for activation-patching-style localization.
# full_stream_results = full_stream_interchange_baseline("NegP")
# plot_full_stream_baseline(full_stream_results, "NegP full-stream interchange")

### Wrong-variable and shuffled-label controls

These controls are useful for the final report.

- **Shuffled labels:** same base/source pairs, random target labels. DAS should not generalize.
- **Wrong variable:** train DAS on one high-level variable but evaluate on counterfactuals for another. This tests whether a subspace is variable-specific or just a generic output hack.

In [ ]:
def train_das_with_raw_data(
    train_raw,
    test_raw,
    metadata_variable_name,
    layer=None,
    subspace_dim=None,
    position=None,
    train_dir=PROJECT_TRAIN_DIR,
    epochs=None,
    lr=None,
    batch_size=None,
    seed=42,
    shuffle_train_labels=False,
):
    layer = EXP["das_layer"] if layer is None else layer
    subspace_dim = EXP["das_subspace_dim"] if subspace_dim is None else subspace_dim
    epochs = epochs or EXP["das_epochs"]
    lr = lr or EXP["das_lr"]
    batch_size = batch_size or EXP["das_batch_size"]

    _, tokenizer, base_model = load_factual_model(train_dir)
    train_dataset = preprocess_counterfactual(train_raw, tokenizer)
    test_dataset = preprocess_counterfactual(test_raw, tokenizer)

    if shuffle_train_labels:
        train_dataset = shuffle_counterfactual_labels(train_dataset, seed=seed)

    intervenable = make_intervenable_model(base_model, layer=layer, subspace_dim=subspace_dim)
    untrained_eval = evaluate_intervenable(intervenable, test_dataset, batch_size=batch_size, position=position)

    intervenable = train_intervenable_on_dataset(
        intervenable,
        train_dataset,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        position=position,
    )
    test_eval = evaluate_intervenable(intervenable, test_dataset, batch_size=batch_size, position=position)

    row = {
        "metadata_variable": metadata_variable_name,
        "layer": layer,
        "subspace_dim": subspace_dim,
        "position": MAX_LENGTH - 2 if position is None else position,
        "majority_baseline": majority_counterfactual_accuracy(test_raw),
        "untrained_rotation_accuracy": untrained_eval["accuracy"],
        "das_accuracy": test_eval["accuracy"],
        "shuffle_train_labels": shuffle_train_labels,
    }
    return row, intervenable

def run_wrong_variable_control(
    target_variable="NegP",
    wrong_train_variable="N_O",
    layer=None,
    subspace_dim=None,
    position=None,
    seed=2024,
):
    train_wrong = generate_counterfactual_data(wrong_train_variable, n=EXP["das_train_n"], seed=seed)
    test_target = generate_counterfactual_data(target_variable, n=EXP["das_test_n"], seed=seed + 1)

    row, _ = train_das_with_raw_data(
        train_raw=train_wrong,
        test_raw=test_target,
        metadata_variable_name=f"train_{wrong_train_variable}_eval_{target_variable}",
        layer=layer,
        subspace_dim=subspace_dim,
        position=position,
        seed=seed,
    )
    display(pd.DataFrame([row]))
    return row

# Uncomment after the main run.
# wrong_var_control = run_wrong_variable_control(target_variable="NegP", wrong_train_variable="N_O")

### Suggested project runs

After the smoke test works and factual accuracy is high, these are the runs I would use for the report:

```python
# 1. Variable hierarchy
variable_results = run_variable_comparison(
    variables=["QP_S", "NegP", "N_P_O", "N_H_O", "N_O", "NP_O", "Q_S"],
    shuffle_control=True,
)

# 2. Layer sweeps for representative variables
layer_dfs = []
for v in ["N_P_O", "N_O", "NegP", "QP_S"]:
    layer_dfs.append(run_layer_sweep(v, layers=[0, 2, 4, 6, 8, 10, 11], subspace_dim=128))
layer_df = pd.concat(layer_dfs)
plot_layer_sweep(layer_df, "DAS layer sweep across NLI variables")

# 3. Dimension sweeps
dim_dfs = []
for v in ["N_O", "NegP", "QP_S"]:
    dim_dfs.append(run_dimension_sweep(v, dims=[1, 4, 8, 16, 32, 64, 128, 256], layer=10))
dim_df = pd.concat(dim_dfs)
plot_dimension_sweep(dim_df, "DAS dimension sweep across NLI variables")

# 4. Full-stream localization baseline
full_dfs = []
for v in ["N_O", "NegP", "QP_S"]:
    full_dfs.append(full_stream_interchange_baseline(v, layers=list(range(12))))
full_df = pd.concat(full_dfs)
plot_full_stream_baseline(full_df, "Full-stream interchange baseline")
```